# Scan2Stage — M3/M4 Room + Object Candidates
UGScan ZIP/GLB → meters → Z-up → floor Z=0 → clip above 2.5 m → rectangular gallery → structural masking → DBSCAN object candidates.


In [ ]:
REPO_URL='https://github.com/6564200/Scan2Stage.git'
WORKDIR='/content/Scan2Stage'
!rm -rf {WORKDIR}
!git clone --branch main --single-branch {REPO_URL} {WORKDIR}
%cd {WORKDIR}
!bash scripts/colab_bootstrap.sh


## Upload UGScan ZIP or GLB
ZIP is accepted directly. For UGScan GLB the default unit scale is 1.0.


In [ ]:
from google.colab import files
from pathlib import Path
uploaded=files.upload()
name=next(iter(uploaded))
assert Path(name).suffix.lower() in {'.zip','.glb','.gltf','.fbx','.obj'}, 'Upload UGScan ZIP/GLB or another supported mesh'
INPUT=Path('/content/Scan2Stage/data')/Path(name).name
INPUT.parent.mkdir(parents=True, exist_ok=True)
Path(name).replace(INPUT)
print(INPUT)


In [ ]:
import subprocess, shlex
OUT=Path('/content/Scan2Stage/outputs/m3_room')
OUT.mkdir(parents=True, exist_ok=True)
cmd=['scan2stage', str(INPUT), '--output-dir', str(OUT), '--samples', '300000', '--source-up', 'y']
print('Running:', ' '.join(shlex.quote(x) for x in cmd))
proc=subprocess.run(cmd, text=True, capture_output=True)
print(proc.stdout)
if proc.stderr:
    print(proc.stderr)
if proc.returncode != 0:
    raise RuntimeError(f'scan2stage failed with exit code {proc.returncode}')
assert (OUT/'report.json').is_file(), 'scan2stage finished without report.json'


In [ ]:
import json
report=json.loads((OUT/'report.json').read_text())
room=json.loads((OUT/'room_geometry.json').read_text())
objects=json.loads((OUT/'object_candidates.json').read_text())
print('Resolved mesh:', report['resolved_mesh'])
print('Format:', report['format'])
print('Bounds in meters:', report['bounds_meters']['extent'])
print('Floor before translation, m:', room['floor']['z_m'])
print('Working volume:', room['working_volume'])
print('Removed above 2.5 m:', room['removed_above_working_height'])
print('Wall candidates:', len(room['wall_candidates']))
print('Rectangle:', room['rectangle'])
print('Object points:', objects['object_points'])
print('DBSCAN noise points:', objects['dbscan_noise_points'])
print('Candidate count:', objects['candidate_count'])
for c in objects['candidates']:
    print(c['id'], 'center=', [round(x,3) for x in c['center_m']], 'extents=', [round(x,3) for x in c['extents_m']], 'points=', c['point_count'])


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
fp=np.asarray(room['footprint_xy_m'])
if len(fp):
    closed=np.vstack([fp, fp[0]])
    plt.figure(figsize=(8,8))
    plt.plot(closed[:,0], closed[:,1], '-o')
    for c in objects['candidates']:
        x,y,_=c['center_m']
        plt.plot(x,y,'x')
        plt.text(x,y,c['id'])
    plt.axis('equal')
    plt.xlabel('X, m'); plt.ylabel('Y, m'); plt.title('Gallery footprint + object candidates')
    plt.grid(True)


In [ ]:
!pytest -q
